In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('C:/SupplyChainAssistant/data/supply_chain_data.csv')
df_clean = df.copy()
print("Data loaded. Shape:", df_clean.shape)
print("Columns:", df_clean.columns.tolist())

Data loaded. Shape: (100, 24)
Columns: ['Product type', 'SKU', 'Price', 'Availability', 'Number of products sold', 'Revenue generated', 'Customer demographics', 'Stock levels', 'Lead times', 'Order quantities', 'Shipping times', 'Shipping carriers', 'Shipping costs', 'Supplier name', 'Location', 'Lead time', 'Production volumes', 'Manufacturing lead time', 'Manufacturing costs', 'Inspection results', 'Defect rates', 'Transportation modes', 'Routes', 'Costs']


In [2]:
def standardize_column_names(df):
    print("\n--- Step 1: Standardizing Column Names ---")
    
    old_columns = df.columns.tolist()
    
    # Convert to lowercase, strip spaces, replace spaces with underscores
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    
    # Rename the two confusing lead time columns
    rename_map = {}
    if 'lead_times' in df.columns:
        rename_map['lead_times'] = 'supplier_lead_time'
    if 'lead_time' in df.columns:
        rename_map['lead_time'] = 'customer_lead_time'
    
    df = df.rename(columns=rename_map)
    
    new_columns = df.columns.tolist()
    
    # Report changes
    changes = 0
    for old, new in zip(old_columns, new_columns):
        if old != new:
            print(f"  Renamed: '{old}' → '{new}'")
            changes += 1
    
    print(f"\n  Total columns renamed: {changes}")
    print("  ✓ Column names standardized")
    return df

# Test it
df_clean = standardize_column_names(df_clean)
print("\nNew column names:")
print(df_clean.columns.tolist())


--- Step 1: Standardizing Column Names ---
  Renamed: 'Product type' → 'product_type'
  Renamed: 'SKU' → 'sku'
  Renamed: 'Price' → 'price'
  Renamed: 'Availability' → 'availability'
  Renamed: 'Number of products sold' → 'number_of_products_sold'
  Renamed: 'Revenue generated' → 'revenue_generated'
  Renamed: 'Customer demographics' → 'customer_demographics'
  Renamed: 'Stock levels' → 'stock_levels'
  Renamed: 'Lead times' → 'supplier_lead_time'
  Renamed: 'Order quantities' → 'order_quantities'
  Renamed: 'Shipping times' → 'shipping_times'
  Renamed: 'Shipping carriers' → 'shipping_carriers'
  Renamed: 'Shipping costs' → 'shipping_costs'
  Renamed: 'Supplier name' → 'supplier_name'
  Renamed: 'Location' → 'location'
  Renamed: 'Lead time' → 'customer_lead_time'
  Renamed: 'Production volumes' → 'production_volumes'
  Renamed: 'Manufacturing lead time' → 'manufacturing_lead_time'
  Renamed: 'Manufacturing costs' → 'manufacturing_costs'
  Renamed: 'Inspection results' → 'inspection_

In [3]:
def fix_datatypes(df):
    print("\n--- Step 2: Fixing Data Types ---")
    
    fixes = 0
    
    for col in df.columns:
        
        # Only attempt conversion on object columns
        if df[col].dtype == 'object':
            
            # Check if majority of values look numeric
            # Try converting and see how many succeed
            converted = pd.to_numeric(df[col], errors='coerce')
            success_rate = converted.notna().sum() / len(df)
            
            # Only convert if 90% or more values are numeric
            if success_rate >= 0.90:
                df[col] = converted
                print(f"  Converted '{col}' from text to numeric (success rate: {success_rate*100:.1f}%)")
                fixes += 1
                continue
            
            # Check if it looks like a date column
            if any(keyword in col.lower() for keyword in 
                   ['date', 'time', 'day', 'month', 'year']):
                try:
                    df[col] = pd.to_datetime(df[col])
                    print(f"  Converted '{col}' to datetime")
                    fixes += 1
                    continue
                except:
                    pass
    
    if fixes == 0:
        print("  All columns already have correct data types")
    else:
        print(f"\n  Total dtype fixes: {fixes}")
    
    print("  ✓ Data types checked")
    return df

# Test it
df_clean = fix_datatypes(df_clean)
print("\nData types after fixing:")
print(df_clean.dtypes)


--- Step 2: Fixing Data Types ---
  All columns already have correct data types
  ✓ Data types checked

Data types after fixing:
product_type                object
sku                         object
price                      float64
availability                 int64
number_of_products_sold      int64
revenue_generated          float64
customer_demographics       object
stock_levels                 int64
supplier_lead_time           int64
order_quantities             int64
shipping_times               int64
shipping_carriers           object
shipping_costs             float64
supplier_name               object
location                    object
customer_lead_time           int64
production_volumes           int64
manufacturing_lead_time      int64
manufacturing_costs        float64
inspection_results          object
defect_rates               float64
transportation_modes        object
routes                      object
costs                      float64
dtype: object


In [4]:
def remove_duplicates(df):
    print("\n--- Step 3: Removing Duplicates ---")
    
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    removed = before - after
    
    if removed == 0:
        print("  No duplicate rows found")
    else:
        print(f"  Removed {removed} duplicate rows")
        print(f"  Rows before: {before}")
        print(f"  Rows after: {after}")
    
    print("  ✓ Duplicates checked")
    return df

# Test it
df_clean = remove_duplicates(df_clean)


--- Step 3: Removing Duplicates ---
  No duplicate rows found
  ✓ Duplicates checked


In [5]:
def handle_missing_values(df):
    print("\n--- Step 4: Handling Missing Values ---")
    
    # Also treat suspicious text values as missing
    suspicious = ['unknown', 'n/a', 'na', 'none', 'null',
                  'not available', 'missing', 'undefined']
    
    # Replace suspicious text values with NaN first
    for col in df.select_dtypes(include='object').columns:
        mask = df[col].str.lower().str.strip().isin(suspicious)
        count = mask.sum()
        if count > 0:
            df.loc[mask, col] = np.nan
            print(f"  Converted {count} suspicious values to NaN in '{col}'")
    
    fixes = 0
    
    for col in df.columns:
        missing_count = df[col].isnull().sum()
        
        if missing_count > 0:
            missing_pct = (missing_count / len(df)) * 100
            
            # If more than 50% missing suggest dropping
            if missing_pct > 50:
                print(f"  WARNING: '{col}' has {missing_pct:.1f}% missing")
                print(f"  Suggestion: Consider dropping this column")
                choice = input(f"  Drop '{col}'? (yes/no): ").strip().lower()
                if choice == 'yes':
                    df = df.drop(columns=[col])
                    print(f"  ✓ Dropped '{col}'")
                    fixes += 1
                    continue
            
            # Fill numeric columns
            if df[col].dtype in ['int64', 'float64']:
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                print(f"  Filled {missing_count} missing in '{col}' with median {median_val:.2f}")
                fixes += 1
            
            # Fill categorical columns
            elif df[col].dtype == 'object':
                mode_val = df[col].mode()[0]
                df[col] = df[col].fillna(mode_val)
                print(f"  Filled {missing_count} missing in '{col}' with mode '{mode_val}'")
                fixes += 1
    
    if fixes == 0:
        print("  No missing values found")
    
    print("  ✓ Missing values handled")
    return df

# Test it
df_clean = handle_missing_values(df_clean)


--- Step 4: Handling Missing Values ---
  Converted 31 suspicious values to NaN in 'customer_demographics'
  Filled 31 missing in 'customer_demographics' with mode 'Female'
  ✓ Missing values handled


In [6]:
def fix_text_consistency(df):
    print("\n--- Step 5: Fixing Text Consistency ---")
    
    fixes = 0
    
    for col in df.select_dtypes(include='object').columns:
        # Strip extra spaces
        before = df[col].copy()
        df[col] = df[col].str.strip()
        
        # Standardize to title case
        df[col] = df[col].str.title()
        
        # Check if anything changed
        changes = (before != df[col]).sum()
        if changes > 0:
            print(f"  Fixed {changes} inconsistent values in '{col}'")
            fixes += 1
    
    if fixes == 0:
        print("  No text inconsistencies found")
    
    print("  ✓ Text consistency fixed")
    return df

# Test it
df_clean = fix_text_consistency(df_clean)


--- Step 5: Fixing Text Consistency ---
  Fixed 100 inconsistent values in 'product_type'
  Fixed 100 inconsistent values in 'sku'
  Fixed 23 inconsistent values in 'customer_demographics'
  ✓ Text consistency fixed


In [7]:
def detect_outliers(df):
    print("\n--- Step 6: Detecting Outliers ---")
    
    outlier_report = []
    
    for col in df.select_dtypes(include='number').columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < lower) | (df[col] > upper)]
        
        if len(outliers) > 0:
            outlier_report.append({
                'column': col,
                'count': len(outliers),
                'lower_bound': round(lower, 2),
                'upper_bound': round(upper, 2)
            })
            print(f"  '{col}': {len(outliers)} outliers found")
            print(f"    Acceptable range: {lower:.2f} to {upper:.2f}")
    
    if not outlier_report:
        print("  No outliers detected")
    else:
        print(f"\n  NOTE: Outliers are flagged but NOT removed.")
        print(f"  In supply chain, extreme values may be real events.")
        print(f"  Review them manually before deciding to remove.")
    
    print("  ✓ Outlier check complete")
    return df, outlier_report

# Test it
df_clean, outlier_report = detect_outliers(df_clean)


--- Step 6: Detecting Outliers ---
  No outliers detected
  ✓ Outlier check complete


In [8]:
def save_cleaned_data(df):
    print("\n--- Step 7: Saving Cleaned Data ---")
    
    output_path = 'C:/SupplyChainAssistant/outputs/supply_chain_cleaned.csv'
    df.to_csv(output_path, index=False)
    
    print(f"  ✓ Cleaned data saved to:")
    print(f"    {output_path}")
    print(f"  Rows: {df.shape[0]}")
    print(f"  Columns: {df.shape[1]}")
    return output_path

# Test it
save_cleaned_data(df_clean)


--- Step 7: Saving Cleaned Data ---
  ✓ Cleaned data saved to:
    C:/SupplyChainAssistant/outputs/supply_chain_cleaned.csv
  Rows: 100
  Columns: 24


'C:/SupplyChainAssistant/outputs/supply_chain_cleaned.csv'

In [9]:
def run_pipeline(filepath):
    print("=" * 50)
    print("SUPPLY CHAIN DATA CLEANING PIPELINE")
    print("=" * 50)
    
    # Load data
    df = load_data(filepath)
    if df is None:
        return None
    
    # Make a copy - never touch original
    df_clean = df.copy()
    
    # Run all cleaning steps in order
    df_clean = standardize_column_names(df_clean)
    df_clean = fix_datatypes(df_clean)
    df_clean = remove_duplicates(df_clean)
    df_clean = handle_missing_values(df_clean)
    df_clean = fix_text_consistency(df_clean)
    df_clean, outlier_report = detect_outliers(df_clean)
    save_cleaned_data(df_clean)
    
    # Final summary
    print("\n" + "=" * 50)
    print("PIPELINE COMPLETE")
    print("=" * 50)
    print(f"Original shape:  {df.shape}")
    print(f"Cleaned shape:   {df_clean.shape}")
    print(f"Columns removed: {df.shape[1] - df_clean.shape[1]}")
    print(f"Rows removed:    {df.shape[0] - df_clean.shape[0]}")
    print("=" * 50)
    
    return df_clean

# RUN THE FULL PIPELINE
df_final = run_pipeline('C:/SupplyChainAssistant/data/supply_chain_data.csv')

SUPPLY CHAIN DATA CLEANING PIPELINE


NameError: name 'load_data' is not defined